# 온라인 쇼핑 구매 여부 예측 — 분류 미니프로젝트

**모듈 1 · Day 2 · 8교시 / TODO Notebook**

- 데이터: UCI Online Shoppers Purchasing Intention (`online_shoppers_intention.csv`)
- 예측 대상: 구매 여부 `Revenue` (0: 미구매, 1: 구매)
- 개발 방법: **공통 코드 실행 → TODO 직접 구현 → CP-01~08 결과 확인**
- 이 Notebook은 *미완성 실습 템플릿*입니다. 상세 과제는 안내 교안을 참고합니다.

**실행 위치:** `mini-project-01/code/` (데이터: `../data/online_shoppers_intention.csv`)

**주의:** 관측된 세션 정보를 분류하는 실습이며, 방문 초기 실시간 구매 예측을 뜻하지 않습니다.


---
## ① 데이터 탐색 및 클래스 분포 — CP-01
**목표:** 데이터 구조·품질을 확인하고 구매·미구매 비율을 시각화합니다.


### ①-1. 데이터 불러오기 — 공통 코드


In [ ]:
# [공통] 패키지 import 및 파일 경로
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
DATA_PATH = "../data/online_shoppers_intention.csv"

# TODO 1-1. pandas로 CSV 파일을 불러와 df에 저장합니다.
# 힌트: pd.read_csv()
df = ...

# TODO 1-2. 데이터의 행·열 크기와 컬럼별 자료형을 출력합니다.
# 힌트: shape, dtypes
...

In [ ]:
df.head()

In [ ]:
df.describe().round(2)

### ①-2. 데이터 품질·특성 분포 확인 — TODO


In [ ]:
# TODO 1-3. 컬럼별 결측값 수와 중복 행 수를 출력합니다.
# 힌트: df.isna().sum(), df.duplicated().sum()
...

In [ ]:
# [공통] 시각화할 주요 수치형 특성
plot_cols = ["ProductRelated", "ProductRelated_Duration", "PageValues", "ExitRates"]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

# TODO 1-4. 각 컬럼의 히스토그램을 그립니다.
# 힌트: for ax, col in zip(axes, plot_cols), ax.hist(..., bins=40)
for ax, col in zip(axes, plot_cols):
    ...

plt.tight_layout()
plt.show()

### ①-3. 구매 여부 0/1 변환 — TODO


In [ ]:
# [공통] 예측 대상과 데이터 유형 확인
TARGET = "Revenue"

# TODO 1-5. Revenue와 Weekend를 int(0/1)로 변환합니다.
# 힌트: astype(int)
...

print(df[[TARGET, "Weekend"]].dtypes)
df[[TARGET, "Weekend"]].head()

### ①-4. 구매·미구매 분포 분석 — TODO


In [ ]:
# TODO 1-6. Revenue의 클래스별 개수와 비율(%)을 계산합니다.
# 힌트: value_counts(), value_counts(normalize=True), sort_index()
class_counts = ...
class_rates = ...

# [공통] 결과표
summary = pd.DataFrame({
    "count": class_counts.values,
    "rate(%)": (class_rates.values * 100).round(2),
}, index=["미구매 (0)", "구매 (1)"])
summary

In [ ]:
# [공통] 클래스 분포 그래프 형식
plt.figure(figsize=(5, 4))

# TODO 1-7. class_counts를 구매/미구매 막대그래프로 표시합니다.
# 힌트: plt.bar()
...

plt.ylabel("sessions")
plt.title("Class distribution: Revenue")
plt.grid(axis="y", alpha=0.3)
plt.show()

**[CP-01] 완료 기준:** 데이터 크기·결측값·중복 여부를 확인하고, 구매·미구매 비율과 클래스 분포 그래프를 완성합니다.

**기억할 내용:** 클래스 불균형이 심하면 Accuracy만으로 분류 성능을 판단하기 어렵습니다.


---
## ② 계층화 데이터 분할 및 특성 분석 — CP-02
**목표:** 구매 비율을 유지하며 분할하고, Train 데이터에서 구매 관련 특성을 분석합니다.


### ②-1. 입력 특성과 정답 구분 — TODO


In [ ]:
# TODO 2-1. 타깃 Revenue를 입력에서 제외하고 X, y를 만듭니다.
# 힌트: FEATURES는 df의 컬럼명 중 TARGET을 제외한 목록입니다.
FEATURES = ...
X = ...
y = ...

print("입력 특성 수:", len(FEATURES))
print("X:", X.shape, "y:", y.shape)

### ②-2. Train·Validation·Test 계층화 분할 — TODO


In [ ]:
from sklearn.model_selection import train_test_split

# TODO 2-2. 계층화 분할을 구현합니다.
# 1) Test를 전체의 15%로 분리합니다.
# 2) 남은 85%에서 Validation을 전체의 약 15%가 되도록 분리합니다.
#    두 호출 모두 stratify와 random_state=42를 설정합니다.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=..., stratify=..., random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=..., stratify=..., random_state=42
)

# [공통] 분할 결과 점검
for name, part_y in [("Train", y_train), ("Validation", y_valid), ("Test", y_test)]:
    print(f"{name:<11} 행 {len(part_y):>6}개 ({len(part_y) / len(df) * 100:4.1f}%)  "
          f"구매 비율 {part_y.mean() * 100:5.2f}%")
assert len(X_train) + len(X_valid) + len(X_test) == len(df)
assert X_train.index.intersection(X_valid.index).empty
assert X_train.index.intersection(X_test.index).empty
assert X_valid.index.intersection(X_test.index).empty

### ②-3. 방문자 유형별 구매율 분석 — TODO


In [ ]:
# [공통] 분석용 데이터 준비: 입력 + 정답
train_view = X_train.copy()
train_view[TARGET] = y_train

# TODO 2-3. VisitorType별 세션 수와 구매율을 계산하고,
# 구매율 내림차순으로 정렬합니다.
# 힌트: groupby(), agg(sessions="size", purchase_rate="mean")
visitor_rate = ...

visitor_rate.round(3)

In [ ]:
plt.figure(figsize=(6, 4))

# TODO 2-4. 방문자 유형별 구매율을 막대그래프로 표현합니다.
# 힌트: visitor_rate.index, visitor_rate["purchase_rate"]
...

plt.ylabel("purchase rate")
plt.title("Purchase rate by visitor type (Train)")
plt.grid(axis="y", alpha=0.3)
plt.show()

### ②-4. 구매 여부별 행동 특성 비교 — TODO


In [ ]:
# [공통] 비교할 행동 특성
compare_cols = ["ProductRelated", "ProductRelated_Duration", "PageValues", "ExitRates"]

# TODO 2-5. Revenue별 compare_cols의 중앙값을 계산합니다.
# 힌트: train_view.groupby(TARGET)[compare_cols].median()
median_by_class = ...

# [공통] 표에 클래스 이름 표시
median_by_class.index = ["미구매 (0)", "구매 (1)"]
median_by_class.round(3)

**[CP-02] 완료 기준:** Train·Validation·Test의 구매 비율을 확인하고, Train에서 방문자 유형별 구매율과 클래스별 행동 특성의 중앙값을 비교합니다.


---
## ③ 데이터 전처리 — CP-03
**목표:** 특성 유형에 맞는 변환을 구성하고 Train 기준 전처리를 검증합니다.


### ③-1. 범주형·수치형·이진형 구분 — TODO


In [ ]:
# TODO 3-1. 입력 특성을 3종류로 구분합니다.
# 범주형: Month, OperatingSystems, Browser, Region, TrafficType, VisitorType
# 수치형: 페이지 수/체류시간, BounceRates, ExitRates, PageValues, SpecialDay
# 이진형: Weekend
categorical_features = [
    # 여기에 범주형 컬럼명을 작성합니다.
]
numeric_features = [
    # 여기에 수치형 컬럼명을 작성합니다.
]
binary_features = [
    # 여기에 이진형 컬럼명을 작성합니다.
]

# [공통] 누락 또는 중복 분류 검증
all_features = categorical_features + numeric_features + binary_features
assert set(all_features) == set(X_train.columns), "특성 목록을 확인하세요."
assert len(all_features) == len(set(all_features)), "중복된 특성이 있습니다."
print("범주형:", categorical_features)
print("수치형:", numeric_features)
print("이진  :", binary_features)

### ③-2. One-Hot 인코딩·표준화·ColumnTransformer — TODO


In [ ]:
# [공통] 전처리 도구
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer


def make_preprocessor():
    # TODO 3-2. 다음 세 변환을 ColumnTransformer에 결합합니다.
    # ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    # ("num", StandardScaler(), numeric_features)
    # ("bin", "passthrough", binary_features)
    return ...


# [공통] 전처리 객체 생성
preprocessor = make_preprocessor()
preprocessor

### ③-3. Train 기준 전처리 결과 확인 — TODO


In [ ]:
# TODO 3-3. Train에서 전처리를 학습하고, Validation에는 동일한 변환을 적용합니다.
# 힌트: fit_transform(X_train), transform(X_valid)
Xt_train = ...
Xt_valid = ...

# [공통] 변환 결과 확인
print("Train:", X_train.shape, "→", Xt_train.shape)
print("Validation:", X_valid.shape, "→", Xt_valid.shape)
print("변환 데이터에 NaN 있음:", np.isnan(Xt_train).any())

cat_categories = preprocessor.named_transformers_["cat"].categories_
n_cat = sum(len(c) for c in cat_categories)
print("One-Hot 컬럼 수:", n_cat)
print("전체 컬럼 수:", Xt_train.shape[1])

**[CP-03] 완료 기준:** `make_preprocessor()`를 구현하고 Train의 `fit_transform()`, Validation의 `transform()` 결과와 Shape를 확인합니다.


---
## ④ 기준 모델 및 로지스틱 회귀 — CP-04
**목표:** DummyClassifier·LogisticRegression을 학습하고 구매 클래스(1)의 분류 지표를 비교합니다.


### ④-1. 기준 모델·로지스틱 회귀 학습 — TODO


In [ ]:
# [공통] 분류 모델 및 Pipeline 도구
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

# TODO 4-1. 각각 새 make_preprocessor()를 포함하는 Pipeline을 만듭니다.
# 1) DummyClassifier(strategy="most_frequent")
# 2) LogisticRegression(max_iter=1500, random_state=42)
dummy_model = ...
logistic_model = ...

# TODO 4-2. 두 모델 모두 Train에서 fit()합니다.
...

print("모델 학습 완료")

### ④-2. Accuracy·Precision·Recall·F1 평가 함수 — TODO


In [ ]:
# [공통] 분류 평가 도구
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report


def classification_scores(y_true, y_pred):
    """구매 클래스(1)를 양성으로 두고 네 지표를 계산합니다."""
    # TODO 4-3. 각 지표를 계산해 딕셔너리로 반환합니다.
    # Precision·Recall·F1에는 zero_division=0을 사용합니다.
    return {
        "Accuracy": ...,
        "Precision": ...,
        "Recall": ...,
        "F1": ...,
    }


# [공통] 평가 및 결과 요약 함수: 이후 ⑤~⑦에서도 재사용합니다.
def evaluate_classification(model, X, y):
    return classification_scores(y, model.predict(X))


def summarize(name, model):
    tr = evaluate_classification(model, X_train, y_train)
    va = evaluate_classification(model, X_valid, y_valid)
    row = {"Model": name}
    row.update({f"Train {k}": v for k, v in tr.items()})
    row.update({f"Valid {k}": v for k, v in va.items()})
    return row


print(evaluate_classification(logistic_model, X_valid, y_valid))

### ④-3. Train·Validation 성능 비교 — TODO


In [ ]:
# [공통] 다음 교시까지 유지할 모델 등록·결과 누적 구조
results = []
model_registry = {}

# TODO 4-4. 두 모델을 model_registry에 이름으로 등록하고,
# summarize() 결과를 results에 추가합니다.
# 이름: "DummyClassifier", "LogisticRegression"
...

pd.DataFrame(results).round(3)

### ④-4. 혼동행렬 분석 — TODO


In [ ]:
# TODO 4-5. LogisticRegression의 Validation 예측값을 구합니다.
logistic_pred = ...

# TODO 4-6. confusion_matrix()로 TN, FP, FN, TP를 계산합니다.
cm = ...

# [공통] 혼동행렬 결과 표시
cm_df = pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])
display(cm_df)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(classification_report(y_valid, logistic_pred, zero_division=0))

**[CP-04] 완료 기준:** 두 모델의 성능표·혼동행렬을 완성하고 Accuracy와 구매 클래스 F1의 차이를 설명합니다.

**의사결정 ③:** 우선할 평가 지표와 이유를 통합 `adr.md`에 기록합니다.


---
## ⑤ Decision Tree 모델 개선 — CP-05
**목표:** 트리 깊이에 따른 Train·Validation F1 변화를 비교합니다.


### ⑤-1. 깊이별 Decision Tree 학습·평가 — TODO


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# [공통] 비교할 트리 깊이
depths = [3, 6, 12]
tree_models = {}
tree_results = []

# TODO 5-1. 깊이별로 DecisionTreeClassifier를 Pipeline에 넣고 학습합니다.
# 각 모델 이름은 f"DecisionTree(max_depth={d})"를 사용합니다.
for d in depths:
    name = f"DecisionTree(max_depth={d})"
    model = ...

    # TODO 5-2. Train에서 모델 학습
    ...

    # [공통] 모델과 결과 누적
    tree_models[d] = model
    model_registry[name] = model
    row = summarize(name, model)
    tree_results.append(row)
    results.append(row)

# [공통] 깊이별 성능표
tree_df = pd.DataFrame(tree_results)
tree_df["max_depth"] = depths
tree_df[["Model", "Train F1", "Valid F1", "Valid Precision", "Valid Recall"]].round(3)

### ⑤-2. 트리 깊이와 F1 시각화 — TODO


In [ ]:
plt.figure(figsize=(7, 4))

# TODO 5-3. 깊이별 Train F1과 Validation F1을 선 그래프로 표현합니다.
# 힌트: plt.plot(tree_df["max_depth"], tree_df["Train F1"], ...)
...

plt.xticks(depths)
plt.xlabel("max_depth")
plt.ylabel("F1 (purchase class)")
plt.ylim(0, 1)
plt.title("Decision Tree: F1 by max_depth")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**[CP-05] 완료 기준:** 깊이 3·6·12의 Train·Validation F1 비교표와 그래프를 완성합니다.

**선택 심화:** `min_samples_leaf` 설정을 추가로 실험합니다.


---
## ⑥ Random Forest 및 전체 모델 비교 — CP-06
**목표:** Random Forest 두 설정을 실험하고 네 종류 분류 모델의 성능을 비교합니다.


### ⑥-1. Random Forest 설정 A/B 실험 — TODO


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# [공통] Random Forest 실험 설정
rf_configs = {
    "A": {"n_estimators": 50, "max_depth": 10},
    "B": {"n_estimators": 100, "max_depth": 15},
}
rf_models = {}

# TODO 6-1. 설정 A/B에 대해 전처리 + RandomForestClassifier Pipeline을 만듭니다.
# random_state=42, n_jobs=-1을 지정하고 params를 활용합니다.
for key, params in rf_configs.items():
    name = f"RandomForest-{key}(n={params['n_estimators']}, depth={params['max_depth']})"
    model = ...

    # TODO 6-2. 모델을 Train에서 학습합니다.
    ...

    # [공통] 비교용 모델과 결과 등록
    rf_models[key] = model
    model_registry[name] = model
    results.append(summarize(name, model))

pd.DataFrame(results[-2:])[["Model", "Train F1", "Valid F1", "Valid Precision", "Valid Recall"]].round(3)

### ⑥-2. 전체 모델 성능표 작성 — TODO


In [ ]:
# [공통] 전체 실험 결과
results_df = pd.DataFrame(results)

# TODO 6-3. Valid F1 높은 순서로 정렬합니다.
# 힌트: sort_values(), reset_index(drop=True)
results_df = ...

# TODO 6-4. Train F1 - Valid F1 차이를 계산해 새 컬럼에 기록합니다.
results_df["Gap (Train-Valid F1)"] = ...

results_df[[
    "Model", "Train F1", "Valid F1", "Valid Precision", "Valid Recall",
    "Valid Accuracy", "Gap (Train-Valid F1)",
]].round(3)

**[CP-06] 완료 기준:** Dummy·Logistic·Decision Tree·Random Forest의 Validation F1과 Train·Validation 차이를 비교합니다.

**선택 심화:** `class_weight="balanced"` 적용 전후 성능을 비교합니다.


---
## ⑦ 분류 오류 분석 및 최종 모델 선택 — CP-07
**목표:** FP·FN 및 임계값별 지표를 비교하고 최종 모델·임계값을 선택합니다.


### ⑦-1. Validation F1 상위 후보 2개 — TODO


In [ ]:
# TODO 7-1. DummyClassifier를 제외한 Valid F1 상위 모델명 2개를 가져옵니다.
# 힌트: results_df["Model"], .head(2), .tolist()
candidate_names = ...

# [공통] 모델명으로 학습된 모델을 조회합니다.
candidates = {name: model_registry[name] for name in candidate_names}
print("분석 후보:", candidate_names)
assert len(candidates) == 2, "후보 모델 2개를 선택하세요."

### ⑦-2. 후보 모델의 FP·FN 확인 — TODO


In [ ]:
# TODO 7-2. 각 후보의 Validation 예측으로 TN·FP·FN·TP를 계산합니다.
# 힌트: model.predict(X_valid), confusion_matrix(..., labels=[0,1]).ravel()
for name, model in candidates.items():
    pred = ...
    tn, fp, fn, tp = ...
    print(f"{name:<40} TN={tn:>5}  FP={fp:>4}  FN={fn:>4}  TP={tp:>4}")

### ⑦-3. 임계값 0.3·0.5·0.7 비교 — TODO


In [ ]:
# [공통] 비교할 분류 임계값
thresholds = [0.3, 0.5, 0.7]
threshold_records = []

for name, model in candidates.items():
    # TODO 7-3. 각 모델의 구매 클래스(1) 예측 확률을 계산합니다.
    # 힌트: predict_proba(X_valid)[:, 1]
    probability = ...

    for t in thresholds:
        # TODO 7-4. 임계값 t로 확률을 0/1 예측으로 변환합니다.
        predicted = ...

        # TODO 7-5. Precision·Recall·F1 등 지표와 혼동행렬을 계산합니다.
        scores = ...
        tn, fp, fn, tp = ...

        # [공통] 비교용 결과 저장
        threshold_records.append({
            "Model": name, "Threshold": t, **scores,
            "FP": int(fp), "FN": int(fn), "TP": int(tp), "TN": int(tn),
        })

# [공통] 임계값별 결과표
threshold_df = pd.DataFrame(threshold_records)
threshold_df[["Model", "Threshold", "Precision", "Recall", "F1", "FP", "FN"]].round(3)

### ⑦-4. 임계값별 Precision·Recall·F1 시각화 — TODO


In [ ]:
# [공통] 그래프 기본 틀
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), sharey=True)
for ax, (name, part) in zip(axes, threshold_df.groupby("Model", sort=False)):
    # TODO 7-6. Precision·Recall·F1을 임계값에 따라 각각 표시합니다.
    # 힌트: for metric in [...], ax.plot(part["Threshold"], part[metric], ...)
    ...

    ax.set_title(name)
    ax.set_xlabel("decision threshold")
    ax.set_xticks(thresholds)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel("score (purchase class)")
plt.tight_layout()
plt.show()

### ⑦-5. 최종 모델과 임계값 선택 — TODO


In [ ]:
# TODO 7-7. F1, Recall, Precision이 높은 순서대로 정렬합니다.
# 힌트: sort_values([...], ascending=False).reset_index(drop=True)
ranked = ...

ranked[["Model", "Threshold", "F1", "Precision", "Recall", "FP", "FN"]].round(3)

In [ ]:
# TODO 7-8. 위 표와 오류 분석 결과를 보고 최종 모델·임계값을 직접 선택합니다.
# SELECTED_MODEL_NAME은 model_registry에 등록된 모델명과 정확히 일치해야 합니다.
# SELECTED_THRESHOLD는 0.3, 0.5, 0.7 중 하나입니다.
SELECTED_MODEL_NAME = None
SELECTED_THRESHOLD = None

# [공통] 선택값 확인
assert SELECTED_MODEL_NAME in model_registry, "등록된 모델명을 작성하세요."
assert SELECTED_THRESHOLD in thresholds, "실험한 임계값 중 하나를 작성하세요."
assert SELECTED_MODEL_NAME in candidates, "⑦교시 후보 모델 중 하나를 선택하세요."
print("최종 선택 모델:", SELECTED_MODEL_NAME)
print("최종 선택 임계값:", SELECTED_THRESHOLD)

**[CP-07] 완료 기준:** 두 후보의 FP·FN과 임계값별 성능을 비교하고, Validation 결과로 최종 모델·임계값을 결정합니다.

**의사결정 ④:** 최종 선택과 근거를 통합 `adr.md`에 기록합니다.

**선택 심화:** PR 곡선으로 임계값의 영향을 분석합니다.


---
## ⑧ 최종 실행 확인 및 프로젝트 제출 — CP-08
**목표:** 공통 코드로 최종 Test 평가·저장·예측을 확인하고 문서와 GitHub 제출을 마무리합니다.

이번 교시는 **마무리 전용**이며 새로운 모델을 구현하지 않습니다.


### ⑧-1. Train+Validation 재학습·Test 평가 — 공통 코드


In [ ]:
# [공통]
from sklearn.base import clone

X_trainval = pd.concat([X_train, X_valid])
y_trainval = pd.concat([y_train, y_valid])
print("최종 학습 데이터:", X_trainval.shape)

final_model = clone(model_registry[SELECTED_MODEL_NAME])
final_model.fit(X_trainval, y_trainval)

test_probability = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_probability >= SELECTED_THRESHOLD).astype(int)
test_scores = classification_scores(y_test, test_pred)

valid_row = threshold_df[(threshold_df["Model"] == SELECTED_MODEL_NAME) & (threshold_df["Threshold"] == SELECTED_THRESHOLD)].iloc[0]

print("최종 모델:", SELECTED_MODEL_NAME, "/ 임계값:", SELECTED_THRESHOLD)
print()
print(f"{'':<10}{'Test':>8}{'Valid(⑦)':>12}")
for k in ["Accuracy", "Precision", "Recall", "F1"]:
    print(f"{k:<10}{test_scores[k]:>8.3f}{valid_row[k]:>12.3f}")

In [ ]:
# [공통] Test Confusion Matrix
test_cm = pd.DataFrame(
    confusion_matrix(y_test, test_pred, labels=[0, 1]),
    index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"],
)
test_cm

### ⑧-2. 모델·임계값 설정 저장 및 재로딩 — 공통 코드


In [ ]:
# [공통]
import json
import joblib
from pathlib import Path

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)
MODEL_PATH = RESULTS_DIR / "02_final_classifier.joblib"
CONFIG_PATH = RESULTS_DIR / "02_model_config.json"

joblib.dump(final_model, MODEL_PATH, compress=3)
config = {
    "model_name": SELECTED_MODEL_NAME,
    "decision_threshold": SELECTED_THRESHOLD,
    "positive_class": 1,
    "features": FEATURES,
}
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", MODEL_PATH, f"({MODEL_PATH.stat().st_size / 1024 / 1024:.1f} MB)")
print("저장:", CONFIG_PATH)

loaded_model = joblib.load(MODEL_PATH)
loaded_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
loaded_pred = (loaded_model.predict_proba(X_test)[:, 1] >= loaded_config["decision_threshold"]).astype(int)
print("저장 전후 예측 일치:", np.array_equal(test_pred, loaded_pred))

### ⑧-3. 구매 여부 예측 함수 확인 — 공통 코드


In [ ]:
# [공통]
def predict_purchase(model, config, record):
    """세션 한 건(dict)을 받아 구매 확률과 구매 여부(0/1)를 반환합니다."""
    row = pd.DataFrame([record])[config["features"]]
    probability = float(model.predict_proba(row)[0, 1])
    return {
        "purchase_probability": round(probability, 4),
        "prediction": int(probability >= config["decision_threshold"]),
        "threshold": config["decision_threshold"],
    }

# 예시 1: Test 세션 한 건 그대로
example = X_test.iloc[0].to_dict()
print("예시 1 (Test 첫 세션)        →", predict_purchase(loaded_model, loaded_config, example),
      "/ 실제:", int(y_test.iloc[0]))

# 예시 2: 같은 세션에서 PageValues만 0으로 바꿈
example_no_value = {**example, "PageValues": 0.0}
print("예시 2 (PageValues = 0)      →", predict_purchase(loaded_model, loaded_config, example_no_value))

# 예시 3: 같은 세션에서 PageValues를 크게 올림
example_high_value = {**example, "PageValues": 50.0}
print("예시 3 (PageValues = 50)     →", predict_purchase(loaded_model, loaded_config, example_high_value))

### ⑧-4. 비교 결과 CSV 2개 저장 — 공통 코드


In [ ]:
# [공통] 주요 실험 결과 CSV 저장 — 총 2개
results_df.round(4).to_csv(
    RESULTS_DIR / "02_model_comparison.csv", index=False
)
threshold_df.round(4).to_csv(
    RESULTS_DIR / "02_threshold_comparison.csv", index=False
)
print("저장:", sorted(p.name for p in RESULTS_DIR.iterdir() if p.name.startswith("02_")))

**[CP-08] 완료 기준:** Test 성능·모델 재로딩·예측 결과를 확인하고, Analysis 2개·통합 ADR(③·④)·명세서·GitHub 제출을 마무리합니다.

**마지막 확인:** 커널 재시작 후 **모든 셀 실행**으로 정상 동작하는지 확인합니다. Test 결과로 모델·임계값을 다시 선택하지 않습니다.

**제출 파일 및 절차:** 과제 안내 교안을 참고합니다.
